# 1. Calculate and save data

This is the single numerical pipeline for the SAXS/PCI study. Shared geometry,
I/O, Gaussian-beam construction, and voxelized-wire projection are defined
once. Each section then keeps only its genuinely distinct method.

Execution order follows the data dependencies:

1. Clean Gaussian SAXS detector scan.
2. SAXS asymmetry reductions, divergence scan, and PCI/SAXS gallery data.
3. Clean detector-propagated PCI and target-exit scan.
4. Realistic-profile SAXS/PCI scan.
5. Gaussian experimental-comparison PCI and transmission data.

The clean SAXS and PCI workflows include an additional target-plane FWHM = 6 µm case.

Notebook 01 performs no plotting. It writes tables, TIFFs, and compact plot-data
bundles. Notebook 02 reads those artifacts and creates every figure without
repeating a propagation. Set `FORCE_RECALCULATE = True` only when existing
compatible TIFFs/tables must be regenerated.

All artifacts for one boundary model are isolated under
`outputs/sigmaL_<left>nm_sigmaR_<right>nm/`. To calculate another model, edit
`WIRE_SIGMA_LEFT_NM` and `WIRE_SIGMA_RIGHT_NM` identically in notebooks 01 and 02.


## Shared setup and numerical building blocks


In [ ]:
from pathlib import Path
import gc
import json
import os
import pickle
import shutil
import sys

import numpy as np
from PIL import Image
from scipy.interpolate import interp1d
from scipy.ndimage import affine_transform, gaussian_filter1d, shift as nd_shift
### change the directory to the SAXS_asymmetry folder if necessary#############
###########################################################################
START_DIR = Path.cwd()
if (START_DIR / "PCI.py").exists():
    SAXS_DIR = START_DIR
elif (START_DIR / "SAXS_asymmetry" / "PCI.py").exists():
    SAXS_DIR = START_DIR / "SAXS_asymmetry"
else:
    raise RuntimeError("Run from the repository root or SAXS_asymmetry/ directory.")
os.chdir(SAXS_DIR)
if str(SAXS_DIR) not in sys.path:
    sys.path.insert(0, str(SAXS_DIR))
######################################################################
import PCI

INPUT_DIR = SAXS_DIR / "inputs"
OUTPUT_ROOT = SAXS_DIR / "outputs"
 
WIRE_SIGMA_LEFT_NM = 4.0  # Fixed left wire sigma in nm
WIRE_SIGMA_RIGHT_NM = 4.0  # Fixed right wire sigma in nm


def sigma_value_tag(value_nm):
    return f"{value_nm:g}".replace(".", "p")

SIGMA_TAG = (
    f"sigmaL_{sigma_value_tag(WIRE_SIGMA_LEFT_NM)}nm_"
    f"sigmaR_{sigma_value_tag(WIRE_SIGMA_RIGHT_NM)}nm"
)
OUTPUT_DIR = OUTPUT_ROOT / SIGMA_TAG
PLOT_DATA_DIR = OUTPUT_DIR / "combined_plot_data"
PLOT_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Shared physical configuration used by the modern 4096-square workflows.
N_GRID = 4096
PIXEL_SIZE = 8 * PCI.nm
PHOTON_ENERGY = 8.2 * PCI.keV
LAMBDA_NM = PCI.HC_KEV_NM / PHOTON_ENERGY
Z_TARGET = 0.25 * PCI.m
Z_SAXS_DETECTOR = 1.31 * PCI.m
Z_PCI_DETECTOR = 6.31 * PCI.m
WIRE_RADIUS = 5 * PCI.um
WIRE_VOXEL_SIZE = 8 * PCI.nm
WIRE_VOXEL_NUMBER = 2000

GRID_CENTER = N_GRID // 2
SPACE_AXIS_NM = (np.arange(N_GRID) - GRID_CENTER) * PIXEL_SIZE
Q_MAX = np.pi / PIXEL_SIZE
Q_AXIS = np.linspace(-Q_MAX, Q_MAX, N_GRID)
Q_STEP = 2 * np.pi / (N_GRID * PIXEL_SIZE)
MARK1300_Q_MIN = 0.10    # Minimum Q value for the Mark1300 reference, because the intensity below this Q are too large
MARK1300_CENTER_HALF_WIDTH_PIXELS = int(np.round(MARK1300_Q_MIN / Q_STEP))

FORCE_RECALCULATE = True  # Reuse valid artifacts; missing cases are still generated

def voxelized_chord_profile(radius, voxel_size, voxel_number):
    """Return the voxelized wire thickness across its diameter."""
    transverse_px = np.arange(voxel_number) - voxel_number // 2
    radius_px = radius / voxel_size
    inside = np.clip(radius_px**2 - transverse_px**2, 0, None)
    return np.where(
        np.abs(transverse_px) <= radius_px,
        (2 * (np.sqrt(inside)) ) * voxel_size,
        0.0,
    )
# wire edge blur profile is gaussian filter with the specified left and right sigmas
def asymmetric_blur_profile(
    profile, *, sigma_left_nm, sigma_right_nm, sample_size_nm,
    center_index=None, truncate=3.0,   # truncate=3 means the Gaussian kernel is truncated at 3 standard deviations
):
    """Blur the negative- and positive-coordinate wire edges independently."""
    profile = np.asarray(profile, dtype=float)


    def blurred(sigma_nm):
        if sigma_nm == 0:
            return profile.copy()
        return gaussian_filter1d(                                  # gaussian_filter1d is used to apply a 1-D Gaussian blur to the profile
            profile, sigma=sigma_nm / sample_size_nm, mode="constant",
            cval=0.0, truncate=truncate,
        )
    # Apply the asymmetric blur to the profile using the specified left and right sigmas
    left_profile = blurred(float(sigma_left_nm))
    right_profile = blurred(float(sigma_right_nm))
    if center_index is None:
        center_index = profile.size // 2
    centered_axis = np.arange(profile.size) - float(center_index)
    return np.where(centered_axis <= 0, left_profile, right_profile)

def source_waist_for_target_fwhm(
    fwhm, *, divergent=True, wavelength=LAMBDA_NM, distance=Z_TARGET
):
    """Return the small/divergent or large/quasi-collimated Gaussian root."""
    target_radius = fwhm / np.sqrt(2 * np.log(2))
    diffraction_term = wavelength * distance / np.pi
    discriminant = max(target_radius**4 - 4 * diffraction_term**2, 0.0)  # Ensure the discriminant is non-negative to avoid complex numbers
    sign = -1 if divergent else 1
    return np.sqrt((target_radius**2 + sign * np.sqrt(discriminant)) / 2)

class ProjectedWire(PCI.wire):
    """Memory-efficient projected wire with independent edge blur."""
    def __init__(self, *args, sigma_left_nm=0.0, sigma_right_nm=0.0, **kwargs):
        self.sigma_left_nm = float(sigma_left_nm)
        self.sigma_right_nm = float(sigma_right_nm)
        self._density1 = None
        super().__init__(*args, **kwargs)

    def _get_projection(self, volume=None):
        if self._density1 is not None:
            return self._density1
        profile = voxelized_chord_profile(
            self.radius1_nm, self.voxel_size, self.voxel_number
        )
        profile = asymmetric_blur_profile(
            profile, sigma_left_nm=self.sigma_left_nm,
            sigma_right_nm=self.sigma_right_nm, sample_size_nm=self.voxel_size,
        )[None, :]

        # Broadcast the 1-D profile to a 2-D projection for the entire wire plane
        projection = np.broadcast_to(
            profile, (self.voxel_number, self.voxel_number)
        ).copy()   # Make a copy to ensure the broadcasted projection is not a view of the original profile
        self._density1 = nd_shift(                        
            projection, np.asarray(self.position_xy) / self.voxel_size,
            order=1, mode="constant", cval=0, prefilter=False,  # Shift the projection to the wire's position in the xy-plane, mode "constant" fills the empty space with 0, order=1 indicates linear interpolation
        )
        return self._density1
    

def make_projected_wire(
    offset_um, *, z_target=Z_TARGET, sigma_left_nm=WIRE_SIGMA_LEFT_NM,
    sigma_right_nm=WIRE_SIGMA_RIGHT_NM,
):
    return ProjectedWire(
        "wire", photon_Energy=PHOTON_ENERGY, radius=WIRE_RADIUS,
        material=PCI.Cu, position=[0, offset_um * PCI.um, z_target],
        voxel_size=WIRE_VOXEL_SIZE, voxel_number=WIRE_VOXEL_NUMBER,
        sigma_left_nm=sigma_left_nm, sigma_right_nm=sigma_right_nm,
    )

def make_xfel(source_waist_nm=None):
    """Monochromatic XFEL on the shared grid; Gaussian when a waist is given."""
    settings = dict(
        pixel_number=N_GRID, pixel_size=PIXEL_SIZE, photon_energy=PHOTON_ENERGY,
        photon_energy_bandwidth=0 * PCI.eV,
        photon_energy_bandwidth_sample_width=0 * PCI.eV,
    )
    if source_waist_nm is None:
        return PCI.XFEL(**settings)
    return PCI.gaussian_XFEL(waist=source_waist_nm, **settings)

def save_float32_tiff(path, array):
    path = Path(path)
    Image.fromarray(np.asarray(array, dtype=np.float32)).save(
        path, format="TIFF", compression="tiff_deflate"
    )

def load_float32_tiff(path):
    with Image.open(path) as image:
        return np.asarray(image, dtype=np.float32).copy()  # Load the TIFF image as a float32 NumPy array and make a copy to ensure dont return a view backed by Pillow’s internal image memory

def central_row_lineout(intensity, row_count, *, dtype=None):
    """Average an even number of detector rows around the center."""
    center = intensity.shape[0] // 2  # intensity.shape[0]  # number of rows: 4096, intensity.shape[1]  # number of columns: 4096, intensity.size  # all pixels: 16,777,216
    half = row_count // 2
    return intensity[center - half:center + half].mean(axis=0, dtype=dtype)

def mark1300_sides(lineout):
    """Sum a lineout outside the excluded |q| < MARK1300_Q_MIN detector center."""
    center = lineout.size // 2
    left = float(lineout[:center - MARK1300_CENTER_HALF_WIDTH_PIXELS].sum())
    right = float(lineout[center + MARK1300_CENTER_HALF_WIDTH_PIXELS:].sum())
    return left, right

def signed_half_weights(axis, *, tolerance, dtype=float):
    """Left/right masks that split the on-axis sample evenly between both halves."""
    left = (axis < 0).astype(dtype)
    right = (axis > 0).astype(dtype)
    on_axis = np.isclose(axis, 0.0, rtol=0.0, atol=tolerance)
    left[on_axis] = right[on_axis] = 0.5
    return left, right

def offset_tag(offset_um, decimals=1):
    return f"{offset_um:+.{decimals}f}".replace("+", "p").replace("-", "m").replace(".", "p")

def simulate_gaussian_saxs_intensity(source_waist_nm, offset_um):
    """Voxelized Cu wire plus PCI.py single-FFT SAXS propagation."""
    xfel = make_xfel(source_waist_nm)
    detector = PCI.jungfrau(
        "SAXS", category="SAXS", XFEL=xfel,
        position=[0, 0, Z_TARGET + Z_SAXS_DETECTOR],
    )
    PCI.Beamline(
        target=make_projected_wire(offset_um), XFEL=xfel, detector=detector
    ).propagate_XFEL_to(detector=detector)
    return np.asarray(xfel.intensity_detector, dtype=np.float32)

def get_gaussian_saxs_tiff(source_waist_nm, offset_um, path):
    """Return a new simulation or load its existing float32 TIFF."""
    path = Path(path)
    if FORCE_RECALCULATE or not path.exists():
        intensity = simulate_gaussian_saxs_intensity(source_waist_nm, offset_um)
        save_float32_tiff(path, intensity)
        return intensity
    return load_float32_tiff(path)

#################################################################
# Experimental pointing profiles
#################################################################
def load_pointing_profiles():   # experimental datasets
    """Mirror each measured half-profile into one normalized signed-q lineout."""
    scales = {528: 50e3, 534: 80e3, 546: 80e3}   # Scale factors for different experimental runs as dictionary
    profiles = {}
    for run, scale in scales.items():
        path = INPUT_DIR / "experimental" / "SAXSData" / f"profiles_cleaned_p3129_r{run:04d}.pickle"
        with path.open("rb") as stream:
            q_positive, left, right = (
                np.nan_to_num(np.asarray(value, dtype=float), nan=0.0)
                for value in pickle.load(stream)[:3]   # Load the first three arrays from the pickle file and convert them to NumPy arrays
            )
        q_values = np.concatenate((np.flip(-q_positive)[:-1], q_positive)) # Construct the full q-axis by mirroring and concatenating the negative and positive parts
        lineout = np.concatenate((np.flip(left)[:-1], right))
        profiles[run] = (q_values, lineout / lineout.max() / scale) # profiles is a dictionary mapping each run number to its corresponding (q_values, normalized lineout) tuple
    return profiles

POINTING_EXPERIMENTAL = load_pointing_profiles()
POINTING_Q_AXIS = POINTING_EXPERIMENTAL[528][0]   # shared experimental q-axis for every simulated pointing lineout
#################################################################
# interpolation for pointing lineout of experiments
def pointing_lineout(lineout):
    resampled = interp1d(Q_AXIS, lineout, fill_value="extrapolate")(POINTING_Q_AXIS)
    return resampled / resampled.max() * 2  # normalizing the pointing profile to have a maximum value of 2

print(f"Shared grid: {N_GRID}x{N_GRID} at {PIXEL_SIZE:g} nm; q step {Q_STEP:.6g} 1/nm")
print(f"Wire blur: left={WIRE_SIGMA_LEFT_NM:g} nm, right={WIRE_SIGMA_RIGHT_NM:g} nm")
print(f"Tagged output directory: {OUTPUT_DIR.resolve()}")


Shared grid: 4096x4096 at 8 nm; q step 0.000191748 1/nm
Wire blur: left=4 nm, right=4 nm
Tagged output directory: C:\Users\feng48\Downloads\PCI_Codebase\PCI_Codebase\SAXS_asymmetry\outputs\sigmaL_4nm_sigmaR_4nm


## 1. Clean Gaussian SAXS detector scan:1. shifted q-axis asymmetry 2. two/ten row lineout 3. divergent and collimated


In [3]:
SAXS_OUTPUT_DIR = OUTPUT_DIR / "saxs"
SAXS_TABLE_DIR = SAXS_OUTPUT_DIR / "tables"
SAXS_TIFF_ROOT = SAXS_OUTPUT_DIR / "detector_tiffs" / f"Data_{PIXEL_SIZE:g}nm"
for directory in (SAXS_TABLE_DIR, SAXS_TIFF_ROOT):
    directory.mkdir(parents=True, exist_ok=True) 

def make_clean_saxs_case(fwhm_um, *, divergent):
    tag = f"{fwhm_um:g}".replace(".", "p")
    geometry = "mark1300" if divergent else "collimated"
    source_waist = source_waist_for_target_fwhm(
        fwhm_um * PCI.um, divergent=divergent
    )
    if divergent:
        label = f"Mark1300 divergent, {fwhm_um:g} µm FWHM at target"
        table_name = f"DivSAXS_asymmetry_Mark1300_{tag}um.dat"
    else:
        label = f"Collimated (large-root), {fwhm_um:g} µm FWHM at target"
        table_name = f"CollimatedSAXS_asymmetry_Mark1300_{tag}um.dat"
    name = f"{geometry}_{tag}um"
    return {
        "name": name,
        "label": label,
        "fwhm_um": float(fwhm_um),
        "source_waist": source_waist,
        "data_dir": SAXS_TIFF_ROOT / name,
        "table_path": SAXS_TABLE_DIR / table_name,
    }

CLEAN_SAXS_CASES = {                                           # CLEAN_SAXS_CASES is a dictionary mapping each case name to its corresponding configuration dictionary
    case["name"]: case for case in (
        make_clean_saxs_case(7.11, divergent=True),
        make_clean_saxs_case(7.11, divergent=False),
        make_clean_saxs_case(6.0, divergent=False),
        make_clean_saxs_case(6.0, divergent=True),
    )
}
CLEAN_SAXS_OFFSET_STEP_UM = 0.48
CLEAN_SAXS_OFFSET_MIN_UM = -10.0
CLEAN_SAXS_OFFSET_MAX_UM = 10.16
CLEAN_SAXS_OFFSETS_UM = np.arange(
    CLEAN_SAXS_OFFSET_MIN_UM,
    CLEAN_SAXS_OFFSET_MAX_UM + 0.5 * CLEAN_SAXS_OFFSET_STEP_UM,
    CLEAN_SAXS_OFFSET_STEP_UM,
    dtype=float,
)
Q_MASK_SHIFT_NM_INV = 0.005
shifted_q_axis = (np.arange(N_GRID) - GRID_CENTER) * Q_STEP
shifted_left_edge = Q_MASK_SHIFT_NM_INV - MARK1300_Q_MIN
shifted_right_edge = Q_MASK_SHIFT_NM_INV + MARK1300_Q_MIN
shifted_left_weights = np.clip((shifted_left_edge - shifted_q_axis) / Q_STEP + 0.5, 0.0, 1.0)  # Compute the left-side weights for the shifted q-axis, between edge is 1, outside is 0
shifted_right_weights = np.clip((shifted_q_axis - shifted_right_edge) / Q_STEP + 0.5, 0.0, 1.0)  # Compute the right-side weights for the shifted q-axis

SHIFTED_MASK_CASES = ("mark1300_7p11um", "mark1300_6um") # shifted_mask_cases is a tuple containing the case names that require shifted-q asymmetry reduction
clean_saxs_records = {}
clean_7p11um_ten_row_pointing = []
shifted_saxs_rows = {case_name: [] for case_name in SHIFTED_MASK_CASES}
clean_profile_bundle = {
    "q_reference": POINTING_Q_AXIS,
    "offsets_um": CLEAN_SAXS_OFFSETS_UM,
}

for case_name, case in CLEAN_SAXS_CASES.items():
    case["data_dir"].mkdir(parents=True, exist_ok=True)  # case is a dictionary 
    records = []   # List to store the offset, left, and right intensity sums for each offset
    pointing_lineouts = []   # List to store the interpolated pointing lineouts for each offset
    print(f"\nClean SAXS: {case['label']}", flush=True) # flush the output immediately to the console
    for index, offset_um in enumerate(CLEAN_SAXS_OFFSETS_UM, start=1):
        detector_path = case["data_dir"] / f"SAXS_Offset{offset_um:.2f}um.tif"
        intensity = get_gaussian_saxs_tiff(                              # intensity is an NumPy array representing the simulated SAXS intensity on the detector
            case["source_waist"], float(offset_um), detector_path
        )
        two_row_lineout = central_row_lineout(intensity, 2)
        left, right = mark1300_sides(two_row_lineout)

        # Interpolate the two-row lineout onto the experimental pointing q-axis for comparison
        plot_lineout = pointing_lineout(two_row_lineout)
        if case_name in SHIFTED_MASK_CASES:
            ten_row_lineout = central_row_lineout(intensity, 10)   # calculated once and reused by both reductions below
            if case_name == "mark1300_7p11um":
                clean_7p11um_ten_row_pointing.append(
                    pointing_lineout(ten_row_lineout).astype(np.float32)
                )
            shifted_fractions = []
            for lineout in (two_row_lineout, ten_row_lineout):
                shifted_left = np.sum(lineout * shifted_left_weights)
                shifted_right = np.sum(lineout * shifted_right_weights)
                shifted_fractions.append(shifted_left / (shifted_left + shifted_right))
            shifted_saxs_rows[case_name].append(
                (float(offset_um), *shifted_fractions) # *shifted_fractions unpacks the list into separate arguments
            )
        records.append((float(offset_um), left, right))
        pointing_lineouts.append(plot_lineout)
        del intensity
        print(f" [{index:02d}/{len(CLEAN_SAXS_OFFSETS_UM)}] {offset_um:+.1f} µm", end="\r")
    records = np.asarray(records, dtype=float)
    clean_saxs_records[case_name] = records
    clean_profile_bundle[f"{case_name}_lineouts"] = np.stack(pointing_lineouts)   # Stack all pointing lineouts for the current case into a single NumPy array and store it in the profile bundle
    np.savetxt(
        case["table_path"],
        np.column_stack((records[:, 0] * 1e3, records[:, 1], records[:, 2])),
        header=("offset_nm left_raw right_raw; Mark1300 two-row extraction; "
                f"{case['label']}"),
    )
    print(
        f"  saved {len(records)} offsets; source radius "
        f"{case['source_waist'] / PCI.um:.6f} µm -> {case['table_path'].name}"
    )

np.savez(PLOT_DATA_DIR / "clean_saxs_profiles.npz", **clean_profile_bundle)
print(f"Clean SAXS plot data: {PLOT_DATA_DIR / 'clean_saxs_profiles.npz'}")



Clean SAXS: Mark1300 divergent, 7.11 µm FWHM at target
Rayleigh [m]:  0.0942016814597633
Radius [um]:  285495.8271593869 0.15120024068359483
the beam waist on the target plane is  6.039 um
the FWHM beam waist on the target plane is  7.110 um
the target postion is 0.250 m
complex q parameter is (249999999.99999997+94201681.4597633j)
mu is 4.0906392887346616e-05 [1/um]
The position is 1310000000.000


"Sorry, due to the detector type 'SAXS', bandwidth not taken into account in propagation from last target to detector!"


Rayleigh [m]:  0.0942016814597633
Radius [um]:  285495.8271593869 0.15120024068359483
the beam waist on the target plane is  6.039 um
the FWHM beam waist on the target plane is  7.110 um
the target postion is 0.250 m
complex q parameter is (249999999.99999997+94201681.4597633j)
mu is 4.0906392887346616e-05 [1/um]
The position is 1310000000.000


"Sorry, due to the detector type 'SAXS', bandwidth not taken into account in propagation from last target to detector!"


Rayleigh [m]:  0.0942016814597633
Radius [um]:  285495.8271593869 0.15120024068359483
the beam waist on the target plane is  6.039 um
the FWHM beam waist on the target plane is  7.110 um
the target postion is 0.250 m
complex q parameter is (249999999.99999997+94201681.4597633j)
mu is 4.0906392887346616e-05 [1/um]
The position is 1310000000.000


"Sorry, due to the detector type 'SAXS', bandwidth not taken into account in propagation from last target to detector!"


... (335 outputs left out)

"Sorry, due to the detector type 'SAXS', bandwidth not taken into account in propagation from last target to detector!"


Rayleigh [m]:  0.16837868596938366
Radius [um]:  363405.5275551053 0.15120024068359483
the beam waist on the target plane is  5.096 um
the FWHM beam waist on the target plane is  6.000 um
the target postion is 0.250 m
complex q parameter is (249999999.99999997+168378685.96938363j)
mu is 4.0906392887346616e-05 [1/um]
The position is 1310000000.000


"Sorry, due to the detector type 'SAXS', bandwidth not taken into account in propagation from last target to detector!"


  saved 43 offsets; source radius 2.846721 µm -> DivSAXS_asymmetry_Mark1300_6um.dat
Clean SAXS plot data: c:\Users\feng48\Downloads\PCI_Codebase\PCI_Codebase\SAXS_asymmetry\outputs\sigmaL_4nm_sigmaR_4nm\combined_plot_data\clean_saxs_profiles.npz


## 2. SAXS asymmetry methods and gallery data 1. SAXS vs Q for different wire offsets 2. movie


In [4]:
METHODS_OUTPUT_DIR = OUTPUT_DIR / "methods"
METHODS_TABLE_DIR = METHODS_OUTPUT_DIR / "tables"
METHODS_MOVIE_DIR = METHODS_OUTPUT_DIR / "movie" / "7p11um"
for directory in (METHODS_TABLE_DIR, METHODS_MOVIE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

METHODS_SOURCE_WAIST = CLEAN_SAXS_CASES["mark1300_7p11um"]["source_waist"]
METHODS_OFFSETS_UM = CLEAN_SAXS_OFFSETS_UM

def make_methods_record(offset_um, left, right, pointing):
    return {
        "offset_um": float(offset_um),
        "mark1300_left": left,
        "mark1300_right": right,
        "mark1300_frac": left / (left + right),
        "pointing_lineout": np.asarray(pointing, dtype=np.float32),
    }

def methods_record_from_intensity(intensity, offset_um):
    left, right = mark1300_sides(central_row_lineout(intensity, 2))
    pointing = pointing_lineout(central_row_lineout(intensity, 10))
    return make_methods_record(offset_um, left, right, pointing)

# Method 1 uses the already-calculated Clean SAXS 7.11µm detector TIFFs.
clean_7p11um_case = CLEAN_SAXS_CASES["mark1300_7p11um"]
clean_7p11um_records = clean_saxs_records["mark1300_7p11um"]
methods_records = [
    make_methods_record(offset_um, left, right, pointing)
    for (offset_um, left, right), pointing
    in zip(clean_7p11um_records, clean_7p11um_ten_row_pointing)
]
methods_table_path = METHODS_TABLE_DIR / "DivSAXS_asymmetry_Methods_7p11um.dat"

np.savetxt(
    methods_table_path,
    np.column_stack((                                   # Stack the offsets, left, and right intensities into columns for saving
        METHODS_OFFSETS_UM * 1e3,
        [r["mark1300_left"] for r in methods_records],   # r["mark1300_left"] for each record in the methods_records list
        [r["mark1300_right"] for r in methods_records],
    )),
    header="offset_nm left_raw right_raw; two-row Mark1300 reduction of Clean SAXS 7p11um TIFFs",
)
np.savez(
    PLOT_DATA_DIR / "methods_pointing_profiles.npz",
    q=POINTING_Q_AXIS,
    offsets_um=METHODS_OFFSETS_UM,
    lineouts=np.stack([r["pointing_lineout"] for r in methods_records]),
)
print(f"Methods offset table: {methods_table_path}")


Methods offset table: c:\Users\feng48\Downloads\PCI_Codebase\PCI_Codebase\SAXS_asymmetry\outputs\sigmaL_4nm_sigmaR_4nm\methods\tables\DivSAXS_asymmetry_Methods_7p11um.dat


### Divergence vary waist with 11 scans 

In [5]:
# Method 2: source-divergence calculations for 6, 9, and 12 µm divergent beams plus the large-root 12 µm case.
divergence_table_path = METHODS_TABLE_DIR / "DivSAXS_divergence_vs_offset_Methods.dat"
DIV_OFFSETS_UM = np.linspace(-10, 10, 50)
DIV_CASES = {
    "6.0 um FWHM": source_waist_for_target_fwhm(6 * PCI.um),
    "9.0 um FWHM": source_waist_for_target_fwhm(9 * PCI.um),
    "12.0 um FWHM": source_waist_for_target_fwhm(12 * PCI.um),
    "collimated": source_waist_for_target_fwhm(12 * PCI.um, divergent=False),
}

divergence_rows = [
    ("7.11 um FWHM", record["offset_um"], record["mark1300_frac"])
    for record in methods_records
]

for case_name, waist_nm in DIV_CASES.items():
    for index, offset_um in enumerate(DIV_OFFSETS_UM, start=1):
        intensity = simulate_gaussian_saxs_intensity(waist_nm, float(offset_um))
        fraction = methods_record_from_intensity(intensity, offset_um)["mark1300_frac"] # methods_record_from_intensity is dictionary containing left, right, fraction, and pointing lineout
        divergence_rows.append((case_name, offset_um, fraction))
        del intensity
        print(f"  divergence {case_name}: {index}/{len(DIV_OFFSETS_UM)}", end="\r")

with divergence_table_path.open("w", encoding="utf-8") as stream:
    stream.write("# case\toffset_um\tmark1300_frac\n")
    for case_name, offset_um, fraction in divergence_rows:
        stream.write(f"{case_name}\t{offset_um:.8f}\t{fraction:.10f}\n")


Rayleigh [m]:  0.16837868596938366
Radius [um]:  363405.5275551053 0.15120024068359483
the beam waist on the target plane is  5.096 um
the FWHM beam waist on the target plane is  6.000 um
the target postion is 0.250 m
complex q parameter is (249999999.99999997+168378685.96938363j)
mu is 4.0906392887346616e-05 [1/um]
The position is 1310000000.000


"Sorry, due to the detector type 'SAXS', bandwidth not taken into account in propagation from last target to detector!"


Rayleigh [m]:  0.16837868596938366
Radius [um]:  363405.5275551053 0.15120024068359483
the beam waist on the target plane is  5.096 um
the FWHM beam waist on the target plane is  6.000 um
the target postion is 0.250 m
complex q parameter is (249999999.99999997+168378685.96938363j)
mu is 4.0906392887346616e-05 [1/um]
The position is 1310000000.000


"Sorry, due to the detector type 'SAXS', bandwidth not taken into account in propagation from last target to detector!"


Rayleigh [m]:  0.16837868596938366
Radius [um]:  363405.5275551053 0.15120024068359483
the beam waist on the target plane is  5.096 um
the FWHM beam waist on the target plane is  6.000 um
the target postion is 0.250 m
complex q parameter is (249999999.99999997+168378685.96938363j)
mu is 4.0906392887346616e-05 [1/um]
The position is 1310000000.000


"Sorry, due to the detector type 'SAXS', bandwidth not taken into account in propagation from last target to detector!"


... (390 outputs left out)

Rayleigh [m]:  2.128905488717613
Radius [um]:  18378954.31956792 0.15120024068359483
the beam waist on the target plane is  10.192 um
the FWHM beam waist on the target plane is  12.000 um
the target postion is 0.250 m
complex q parameter is (249999999.9999999+2128905488.717613j)
mu is 4.0906392887346616e-05 [1/um]
The position is 1310000000.000


"Sorry, due to the detector type 'SAXS', bandwidth not taken into account in propagation from last target to detector!"


Rayleigh [m]:  2.128905488717613
Radius [um]:  18378954.31956792 0.15120024068359483
the beam waist on the target plane is  10.192 um
the FWHM beam waist on the target plane is  12.000 um
the target postion is 0.250 m
complex q parameter is (249999999.9999999+2128905488.717613j)
mu is 4.0906392887346616e-05 [1/um]
The position is 1310000000.000


"Sorry, due to the detector type 'SAXS', bandwidth not taken into account in propagation from last target to detector!"


### Movie

In [6]:
# Method 3 data: reuse the Clean SAXS 7.11µm TIFFs and calculate only the
# distinct direct post-sample PCI images. Figure frames and the GIF are made in 02.
methods_saxs_paths = [
    METHODS_MOVIE_DIR / f"DivSAXS_Offset{offset_um:.2f}um.tif"
    for offset_um in METHODS_OFFSETS_UM
]
methods_pci_paths = [
    METHODS_MOVIE_DIR / f"DivPCI_AfterSample_Offset{offset_um:.2f}um.tif"
    for offset_um in METHODS_OFFSETS_UM
]
methods_xfel = methods_incident = None
if FORCE_RECALCULATE or any(not path.exists() for path in methods_pci_paths):
    methods_xfel = make_xfel(METHODS_SOURCE_WAIST)
    methods_incident = methods_xfel.initialize_to_position(Z_TARGET, reset=False) # initialize_to_position is a library in PCI.py

def methods_pci_exit_intensity(offset_um):
    target = make_projected_wire(offset_um)
    exit_field = methods_xfel.propagate_through_target(
        target, reset=False, field_in=methods_incident, photon_energy=PHOTON_ENERGY
    )
    return np.abs(exit_field)**2

for index, (offset_um, methods_saxs_tiff, methods_pci_tiff) in enumerate(
    zip(METHODS_OFFSETS_UM, methods_saxs_paths, methods_pci_paths), start=1
):
    clean_tiff = clean_7p11um_case["data_dir"] / f"SAXS_Offset{offset_um:.2f}um.tif"
    if FORCE_RECALCULATE or not methods_saxs_tiff.exists():
        shutil.copy2(clean_tiff, methods_saxs_tiff)   # Copy the clean SAXS TIFF to the methods SAXS path if it doesn't already exist
    if FORCE_RECALCULATE or not methods_pci_tiff.exists():
        save_float32_tiff(methods_pci_tiff, methods_pci_exit_intensity(float(offset_um)))   # Save the PCI exit intensity to the methods PCI path if it doesn't already exist
    print(f"Gallery data {index}/{len(METHODS_OFFSETS_UM)}: {offset_um:+.1f} µm")

del methods_incident, methods_xfel
print("Methods data complete: default SAXS propagation was not repeated.")


Rayleigh [m]:  0.0942016814597633
Radius [um]:  285495.8271593869 0.15120024068359483
the beam waist on the target plane is  6.039 um
the FWHM beam waist on the target plane is  7.110 um
the target postion is 0.250 m
complex q parameter is (249999999.99999997+94201681.4597633j)
mu is 4.0906392887346616e-05 [1/um]
Gallery data 1/43: -10.0 µm
mu is 4.0906392887346616e-05 [1/um]
Gallery data 2/43: -9.5 µm
mu is 4.0906392887346616e-05 [1/um]
Gallery data 3/43: -9.0 µm
mu is 4.0906392887346616e-05 [1/um]
Gallery data 4/43: -8.6 µm
mu is 4.0906392887346616e-05 [1/um]
Gallery data 5/43: -8.1 µm
mu is 4.0906392887346616e-05 [1/um]
Gallery data 6/43: -7.6 µm
mu is 4.0906392887346616e-05 [1/um]
Gallery data 7/43: -7.1 µm
mu is 4.0906392887346616e-05 [1/um]
Gallery data 8/43: -6.6 µm
mu is 4.0906392887346616e-05 [1/um]
Gallery data 9/43: -6.2 µm
mu is 4.0906392887346616e-05 [1/um]
Gallery data 10/43: -5.7 µm
mu is 4.0906392887346616e-05 [1/um]
Gallery data 11/43: -5.2 µm
mu is 4.0906392887346616e

## 3. Clean detector-propagated PCI and target-exit scan


In [7]:
CLEAN_PCI_DIR = OUTPUT_DIR / "clean_pci"
CLEAN_PCI_DIR.mkdir(parents=True, exist_ok=True)

COPPER_ATTENUATION_LENGTH_NM = 1 / (
    PCI.Cu().get_n_refraction_beta(PHOTON_ENERGY) * PHOTON_ENERGY * 10
)

def angular_spectrum_propagate(field, distance_m):
    """Exact angular-spectrum free-space propagation over `distance_m` metres."""
    pixel_size_m = PIXEL_SIZE / PCI.m
    wavenumber_m = 2 * np.pi / (LAMBDA_NM / PCI.m)
    spatial_frequency = np.fft.fftfreq(N_GRID, d=pixel_size_m)
    kx, ky = np.meshgrid(
        2 * np.pi * spatial_frequency, 2 * np.pi * spatial_frequency, indexing="xy"
    )    # Create a 2D meshgrid of spatial frequencies for the angular spectrum propagation
    dispersion = wavenumber_m**2 - kx**2 - ky**2
    root = np.sqrt(np.abs(dispersion))
    kz = np.where(dispersion >= 0, root, 1j * root)   # evanescent components decay instead of oscillating
    return np.fft.ifft2(np.fft.fft2(field) * np.exp(1j * kz * distance_m))

def simulate_clean_pci(fwhm_um):
    xfel = make_xfel(source_waist_for_target_fwhm(fwhm_um * PCI.um))
    incident = xfel.initialize_to_position(Z_TARGET, reset=False)
    target = make_projected_wire(0.0) # make projected wire is an array including wire radius, voxel size, and voxel number, material, and position, absorption and phase shift
    exit_field = xfel.propagate_through_target(
        target, reset=False, field_in=incident,
        photon_energy=PHOTON_ENERGY,
    )
    wire_projection_nm = target._get_projection()

    thickness_nm = xfel._interpolate_target2beam(
        wire_projection_nm,
        target,
    )
    detector_field = angular_spectrum_propagate(exit_field, Z_PCI_DETECTOR / PCI.m)
    return incident, thickness_nm, exit_field, detector_field

clean_pci_fields_path = PLOT_DATA_DIR / "clean_pci_fields.npz"
clean_pci_detector_path = CLEAN_PCI_DIR / "clean_pci_detector_intensity.tif"
CLEAN_PCI_CASES = {
    "7p11um": {
        "fwhm_um": 7.11,
        "fields_path": clean_pci_fields_path,
        "detector_path": clean_pci_detector_path,
        "table_path": CLEAN_PCI_DIR / "pci_exit_left_right_vs_offset.dat",
        "raw_table_path": CLEAN_PCI_DIR / "clean_pci_exit_left_right_raw_vs_offset.dat",
    },
    "6um": {
        "fwhm_um": 6.0,
        "fields_path": PLOT_DATA_DIR / "clean_pci_fields_6um.npz",
        "detector_path": CLEAN_PCI_DIR / "clean_pci_detector_intensity_6um.tif",
        "table_path": CLEAN_PCI_DIR / "pci_exit_left_right_vs_offset_6um.dat",
        "raw_table_path": CLEAN_PCI_DIR / "clean_pci_exit_left_right_raw_vs_offset_6um.dat",
    },
}
pci_extent_um = np.array([
    SPACE_AXIS_NM[0], SPACE_AXIS_NM[-1], SPACE_AXIS_NM[0], SPACE_AXIS_NM[-1],
]) / PCI.um
for case_name, case in CLEAN_PCI_CASES.items():
    if FORCE_RECALCULATE or not (
        case["fields_path"].exists() and case["detector_path"].exists()
    ):
        pci_incident, pci_thickness_nm, pci_exit, pci_detector = simulate_clean_pci(
            case["fwhm_um"]
        )
        case["incident_intensity"] = np.abs(pci_incident)**2
        np.savez(
            case["fields_path"],
            incident_intensity=case["incident_intensity"],
            thickness_um=pci_thickness_nm / PCI.um,
            exit_intensity=np.abs(pci_exit)**2,
            extent_um=pci_extent_um,
            fwhm_um=case["fwhm_um"],
        )
        save_float32_tiff(case["detector_path"], np.abs(pci_detector)**2)
        del pci_incident, pci_thickness_nm, pci_exit, pci_detector
        gc.collect()
    else:
        with np.load(case["fields_path"]) as fields:
            case["incident_intensity"] = fields["incident_intensity"]
        print(f"Reusing clean PCI {case_name} field bundle and detector TIFF")


Rayleigh [m]:  0.0942016814597633
Radius [um]:  285495.8271593869 0.15120024068359483
the beam waist on the target plane is  6.039 um
the FWHM beam waist on the target plane is  7.110 um
the target postion is 0.250 m
complex q parameter is (249999999.99999997+94201681.4597633j)
mu is 4.0906392887346616e-05 [1/um]
Rayleigh [m]:  0.16837868596938366
Radius [um]:  363405.5275551053 0.15120024068359483
the beam waist on the target plane is  5.096 um
the FWHM beam waist on the target plane is  6.000 um
the target postion is 0.250 m
complex q parameter is (249999999.99999997+168378685.96938363j)
mu is 4.0906392887346616e-05 [1/um]


In [8]:
def scan_clean_pci_exit(wire_offsets_nm, incident_intensity):
    total_before = float(incident_intensity.sum())
    center_index = incident_intensity.shape[0] // 2
    incident_lineout = incident_intensity[center_index]
    y_factor = (
        incident_intensity[:, center_index].sum()
        / incident_intensity[center_index, center_index]
    )
    axis_nm = SPACE_AXIS_NM
    target_axis_nm = (
        np.arange(WIRE_VOXEL_NUMBER) - WIRE_VOXEL_NUMBER // 2
    ) * WIRE_VOXEL_SIZE
    centered_chord_nm = voxelized_chord_profile(
        WIRE_RADIUS, WIRE_VOXEL_SIZE, WIRE_VOXEL_NUMBER
    )
    centered_chord_nm = asymmetric_blur_profile(
        centered_chord_nm, sigma_left_nm=WIRE_SIGMA_LEFT_NM,
        sigma_right_nm=WIRE_SIGMA_RIGHT_NM, sample_size_nm=WIRE_VOXEL_SIZE,
    )
    left_weights, right_weights = signed_half_weights(
        axis_nm, tolerance=PIXEL_SIZE / 2
    )
    left = np.empty(len(wire_offsets_nm))
    right = np.empty(len(wire_offsets_nm))
    for index, offset_nm in enumerate(wire_offsets_nm):
        shifted_chord_nm = nd_shift(
            centered_chord_nm, offset_nm / WIRE_VOXEL_SIZE, order=1,
            mode="constant", cval=0, prefilter=False,
        )
        thickness_nm = np.interp(
            axis_nm, target_axis_nm, shifted_chord_nm, left=0.0, right=0.0
        )
        exit_lineout = incident_lineout * np.exp(
            -thickness_nm / COPPER_ATTENUATION_LENGTH_NM
        )
        left[index] = y_factor * np.sum(exit_lineout * left_weights)
        right[index] = y_factor * np.sum(exit_lineout * right_weights)
    return left, right, total_before

clean_pci_offsets_um = np.linspace(-10.0, 10.0, 50)
for case_name, case in CLEAN_PCI_CASES.items():
    clean_pci_left, clean_pci_right, clean_pci_before = scan_clean_pci_exit(
        clean_pci_offsets_um * PCI.um, case.pop("incident_intensity")
    )
    clean_pci_total = clean_pci_left + clean_pci_right
    clean_pci_normalized = np.column_stack((
        clean_pci_left / clean_pci_before,
        clean_pci_right / clean_pci_before,
        clean_pci_total / clean_pci_before,
    ))
    if not np.all(np.isfinite(clean_pci_normalized)):
        raise FloatingPointError(f"Clean PCI {case_name} scan contains non-finite values")
    if np.any(clean_pci_normalized < 0) or np.any(clean_pci_normalized > 1 + 1e-12):
        raise ValueError(f"Clean PCI {case_name} transmission must remain between 0 and 1")
    np.savetxt(
        case["raw_table_path"],
        np.column_stack((
            clean_pci_offsets_um, clean_pci_left, clean_pci_right, clean_pci_total,
        )),
        header=("offset_um raw_left_exit_intensity_au raw_right_exit_intensity_au "
                f"raw_total_exit_intensity_au; FWHM={case['fwhm_um']:g} um"),
    )
    np.savetxt(
        case["table_path"],
        np.column_stack((clean_pci_offsets_um, clean_pci_normalized)),
        header=("offset_um left_exit_over_total_before right_exit_over_total_before "
                f"total_exit_over_total_before; FWHM={case['fwhm_um']:g} um"),
    )
    print(f"Clean PCI {case_name}: {case['table_path']}")


Clean PCI 7p11um: c:\Users\feng48\Downloads\PCI_Codebase\PCI_Codebase\SAXS_asymmetry\outputs\sigmaL_4nm_sigmaR_4nm\clean_pci\pci_exit_left_right_vs_offset.dat
Clean PCI 6um: c:\Users\feng48\Downloads\PCI_Codebase\PCI_Codebase\SAXS_asymmetry\outputs\sigmaL_4nm_sigmaR_4nm\clean_pci\pci_exit_left_right_vs_offset_6um.dat


### Save PCI shifted-q asymmetry reduction

In [9]:
# Figure-3 detector-center correction was reduced during the clean SAXS pass,
# so its TIFFs do not need to be loaded and averaged a second time here.
saxs_7_shifted = np.asarray(shifted_saxs_rows["mark1300_7p11um"])
np.savetxt(
    CLEAN_PCI_DIR / "mark1300_7p11um_shifted_q_mask_left_fraction.dat",
    saxs_7_shifted,
    header=("offset_7p11_um two_row_7p11 ten_row_7p11; "
            f"q_shift_nm_inv={Q_MASK_SHIFT_NM_INV} q_min_nm_inv={MARK1300_Q_MIN}"),
)
saxs_6_shifted = np.asarray(shifted_saxs_rows["mark1300_6um"])
np.savetxt(
    CLEAN_PCI_DIR / "mark1300_6p0um_shifted_q_mask_left_fraction.dat",
    saxs_6_shifted,
    header=("offset_6p0_um two_row_6p0 ten_row_6p0; "
            f"q_shift_nm_inv={Q_MASK_SHIFT_NM_INV} q_min_nm_inv={MARK1300_Q_MIN}"),
)
print("Clean PCI data complete.")


Clean PCI data complete.


## 4. Realistic-profile SAXS/PCI scan


In [10]:
REALISTIC_DIR = OUTPUT_DIR / "realistic_profile"
REALISTIC_TABLE_DIR = REALISTIC_DIR / "tables"
REALISTIC_TIFF_DIR = REALISTIC_DIR / "detector_tiffs"
for directory in (REALISTIC_DIR, REALISTIC_TABLE_DIR, REALISTIC_TIFF_DIR):
    directory.mkdir(parents=True, exist_ok=True)

PROFILE_PATH = INPUT_DIR / "profiles" / "RealisticProfile_10nm.tif"
PROFILE_PIXEL_SIZE = 10 * PCI.nm
REALISTIC_OFFSETS_UM = CLEAN_SAXS_OFFSETS_UM
REALISTIC_Q_MIN = 0.10

with Image.open(PROFILE_PATH) as image:
    profile_native = np.array(image.convert("F"), dtype=np.float32)   # np.array copies, so the result stays writable for the in-place cleanup below
np.nan_to_num(profile_native, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
np.maximum(profile_native, 0.0, out=profile_native)
index_scale = float(PIXEL_SIZE / PROFILE_PIXEL_SIZE)
profile_raw = affine_transform(
    profile_native,
    matrix=np.eye(2) * index_scale,
    offset=np.asarray(profile_native.shape, dtype=float) / 2 - index_scale * N_GRID / 2,
    output_shape=(N_GRID, N_GRID), order=1, mode="constant", cval=0.0,
    prefilter=False, output=np.float32,
)
realistic_incident = (
    profile_raw / np.sum(profile_raw, dtype=np.float64)
).astype(np.float32, copy=False)
np.save(PLOT_DATA_DIR / "realistic_incident_intensity.npy", realistic_incident)
realistic_xfel = PCI.XFEL(
    pixel_number=N_GRID, pixel_size=PIXEL_SIZE, photon_energy=PHOTON_ENERGY,
    photon_energy_bandwidth=0 * PCI.eV,
    photon_energy_bandwidth_sample_width=0 * PCI.eV,
)
phase_source_waist = CLEAN_SAXS_CASES["mark1300_7p11um"]["source_waist"]
phase_rayleigh_range = np.pi * phase_source_waist**2 / LAMBDA_NM
phase_wavefront_radius = (Z_TARGET**2 + phase_rayleigh_range**2) / Z_TARGET
phase_1d = np.exp(
    -0.5j * realistic_xfel.get_k(photon_energy=PHOTON_ENERGY)
    * SPACE_AXIS_NM**2 / phase_wavefront_radius
).astype(np.complex64)
realistic_field_at_target = np.sqrt(realistic_incident).astype(np.complex64)
realistic_field_at_target *= phase_1d[:, None]
realistic_field_at_target *= phase_1d[None, :]


In [11]:
realistic_pci_table_path = REALISTIC_TABLE_DIR / "realistic_profile_pci_exit_left_right.dat"
realistic_saxs_table_path = REALISTIC_TABLE_DIR / "realistic_profile_saxs_left_right.dat"
realistic_lineout_path = PLOT_DATA_DIR / "realistic_saxs_lineouts.npz"

def realistic_tiff_path(offset_um):
    return REALISTIC_TIFF_DIR / f"SAXS_realistic_offset_{offset_tag(offset_um)}um.tif"

can_reuse_realistic = (
    not FORCE_RECALCULATE
    and realistic_pci_table_path.exists()
    and realistic_saxs_table_path.exists()
    and all(realistic_tiff_path(offset).exists() for offset in REALISTIC_OFFSETS_UM)
)

if can_reuse_realistic:
    pci_table = np.loadtxt(realistic_pci_table_path)
    saxs_table = np.loadtxt(realistic_saxs_table_path)
    realistic_pci_left, realistic_pci_right = pci_table[:, 1], pci_table[:, 2]
    realistic_saxs_left, realistic_saxs_right = saxs_table[:, 1], saxs_table[:, 2]
    realistic_lineouts = []
    for offset_um in REALISTIC_OFFSETS_UM:
        intensity = load_float32_tiff(realistic_tiff_path(offset_um))
        realistic_lineouts.append(
            central_row_lineout(intensity, 10, dtype=np.float64)
        )
        del intensity
else:
    left_q_mask = Q_AXIS < -REALISTIC_Q_MIN
    right_q_mask = Q_AXIS > REALISTIC_Q_MIN
    left_x_weights, right_x_weights = signed_half_weights(
        SPACE_AXIS_NM, tolerance=PIXEL_SIZE / 2, dtype=np.float32
    )
    incident_total = float(np.sum(realistic_incident, dtype=np.float64))
    realistic_pci_left, realistic_pci_right = [], []
    realistic_saxs_left, realistic_saxs_right, realistic_lineouts = [], [], []
    for index, offset_um in enumerate(REALISTIC_OFFSETS_UM, start=1):
        target = make_projected_wire(float(offset_um))
        exit_field = realistic_xfel.propagate_through_target(
            target, reset=False, field_in=realistic_field_at_target,
            photon_energy=PHOTON_ENERGY,
        )
        exit_intensity = np.abs(exit_field)**2
        realistic_pci_left.append(float(np.sum(exit_intensity * left_x_weights[None, :], dtype=np.float64)))
        realistic_pci_right.append(float(np.sum(exit_intensity * right_x_weights[None, :], dtype=np.float64)))
        saxs_field = realistic_xfel.lorentz_propagate_to_position(
            position=Z_SAXS_DETECTOR, field_in=exit_field,
            photon_energy=PHOTON_ENERGY, reset=False,
        )
        saxs_intensity = np.asarray(np.abs(saxs_field)**2, dtype=np.float32)
        save_float32_tiff(realistic_tiff_path(offset_um), saxs_intensity)
        lineout = central_row_lineout(saxs_intensity, 10, dtype=np.float64)
        realistic_lineouts.append(lineout)
        realistic_saxs_left.append(float(lineout[left_q_mask].sum()))
        realistic_saxs_right.append(float(lineout[right_q_mask].sum()))
        del target, exit_field, exit_intensity, saxs_field, saxs_intensity
        gc.collect()
        print(f"Realistic scan {index}/{len(REALISTIC_OFFSETS_UM)}", end="\r")
    realistic_pci_left = np.asarray(realistic_pci_left)
    realistic_pci_right = np.asarray(realistic_pci_right)
    realistic_saxs_left = np.asarray(realistic_saxs_left)
    realistic_saxs_right = np.asarray(realistic_saxs_right)
    realistic_pci_total = realistic_pci_left + realistic_pci_right
    realistic_saxs_total = realistic_saxs_left + realistic_saxs_right
    np.savetxt(
        realistic_pci_table_path,
        np.column_stack((
            REALISTIC_OFFSETS_UM, realistic_pci_left, realistic_pci_right,
            realistic_pci_total,
            realistic_pci_left / realistic_pci_total,
            realistic_pci_total / incident_total,
        )),
        header="offset_um left_exit_raw right_exit_raw total_exit_raw left_over_left_plus_right total_exit_over_incident",
    )
    np.savetxt(
        realistic_saxs_table_path,
        np.column_stack((
            REALISTIC_OFFSETS_UM * 1e3, realistic_saxs_left, realistic_saxs_right,
            realistic_saxs_left / realistic_saxs_total,
        )),
        header="offset_nm left_raw right_raw left_over_left_plus_right; ten-row extraction; abs(qx)>=0.10 nm^-1",
    )

realistic_lineouts = np.stack(realistic_lineouts)
np.savez(
    realistic_lineout_path,
    offsets_um=REALISTIC_OFFSETS_UM,
    q_axis=Q_AXIS,
    lineouts=realistic_lineouts,
)


mu is 4.0906392887346616e-05 [1/um]
The position is 1310000000.000
mu is 4.0906392887346616e-05 [1/um]
The position is 1310000000.000
mu is 4.0906392887346616e-05 [1/um]
The position is 1310000000.000
mu is 4.0906392887346616e-05 [1/um]
The position is 1310000000.000
mu is 4.0906392887346616e-05 [1/um]
The position is 1310000000.000
mu is 4.0906392887346616e-05 [1/um]
The position is 1310000000.000
mu is 4.0906392887346616e-05 [1/um]
The position is 1310000000.000
mu is 4.0906392887346616e-05 [1/um]
The position is 1310000000.000
mu is 4.0906392887346616e-05 [1/um]
The position is 1310000000.000
mu is 4.0906392887346616e-05 [1/um]
The position is 1310000000.000
mu is 4.0906392887346616e-05 [1/um]
The position is 1310000000.000
mu is 4.0906392887346616e-05 [1/um]
The position is 1310000000.000
mu is 4.0906392887346616e-05 [1/um]
The position is 1310000000.000
mu is 4.0906392887346616e-05 [1/um]
The position is 1310000000.000
mu is 4.0906392887346616e-05 [1/um]
The position is 1310000000

## 5. Gaussian experimental-comparison PCI and transmission data 1.PCI compared with SAXS image 2. PCI with experimental data


In [12]:
EXPERIMENTAL_DIR = OUTPUT_DIR / "experimental_comparison"
EXPERIMENTAL_TABLE_DIR = EXPERIMENTAL_DIR / "tables"
EXPERIMENTAL_PCI_TIFF_DIR = EXPERIMENTAL_DIR / "pci_exit_tiffs"
for directory in (EXPERIMENTAL_DIR, EXPERIMENTAL_TABLE_DIR, EXPERIMENTAL_PCI_TIFF_DIR):
    directory.mkdir(parents=True, exist_ok=True)

comparison_case = CLEAN_SAXS_CASES["mark1300_7p11um"]
comparison_xfel = make_xfel(comparison_case["source_waist"])
comparison_field_at_target = comparison_xfel.initialize_to_position(
    Z_TARGET, reset=False
)


Rayleigh [m]:  0.0942016814597633
Radius [um]:  285495.8271593869 0.15120024068359483
the beam waist on the target plane is  6.039 um
the FWHM beam waist on the target plane is  7.110 um
the target postion is 0.250 m
complex q parameter is (249999999.99999997+94201681.4597633j)


In [13]:
# The SAXS half of the comparison is identical to the Clean SAXS 7.11µm
# detector calculation and is therefore reused. Only the analytic-chord PCI
# exit images, which are a genuinely different method, are calculated here.
copper = PCI.Cu()
absorption_coefficient_nm = (
    copper.get_n_refraction_beta(PHOTON_ENERGY) * PHOTON_ENERGY * 10
)
phase_coefficient_nm = copper._get_phase_shift(PHOTON_ENERGY)

def analytic_wire_chord_nm(offset_um):
    offset_nm = offset_um * PCI.um
    transverse_nm = SPACE_AXIS_NM - offset_nm
    shifted_chord = 2 * np.sqrt(
        np.clip(WIRE_RADIUS**2 - transverse_nm**2, 0.0, None)
    )
    return asymmetric_blur_profile(
        shifted_chord, sigma_left_nm=WIRE_SIGMA_LEFT_NM,
        sigma_right_nm=WIRE_SIGMA_RIGHT_NM, sample_size_nm=PIXEL_SIZE,
        center_index=GRID_CENTER + offset_nm / PIXEL_SIZE,
    )

def analytic_pci_exit_intensity(offset_um):
    thickness = np.broadcast_to(analytic_wire_chord_nm(offset_um), (N_GRID, N_GRID))
    factor = np.exp(
        (-0.5 * absorption_coefficient_nm - 1j * phase_coefficient_nm) * thickness
    )
    return np.abs(comparison_field_at_target * factor)**2

# Generate analytic PCI on the same 0.48 um grid as the reused Clean SAXS TIFFs.
comparison_offsets_um = CLEAN_SAXS_OFFSETS_UM.copy()
comparison_pci_paths = [
    EXPERIMENTAL_PCI_TIFF_DIR / f"analytic_pci_exit_offset_{offset_tag(offset)}um.tif"
    for offset in comparison_offsets_um
]
comparison_saxs_paths = [
    comparison_case["data_dir"]
    / f"SAXS_Offset{(-0.0 if np.isclose(offset, 0.0) else offset):.2f}um.tif"
    for offset in comparison_offsets_um
]
for index, (offset_um, pci_path) in enumerate(
    zip(comparison_offsets_um, comparison_pci_paths), start=1
):
    if FORCE_RECALCULATE or not pci_path.exists():
        save_float32_tiff(pci_path, analytic_pci_exit_intensity(float(offset_um)))
    print(f"Experimental comparison data {index}/{len(comparison_offsets_um)}")

transmission_offsets_um = np.linspace(-10.0, 10.0, 49)
incident_intensity = np.abs(comparison_field_at_target)**2
incident_column_power = np.sum(incident_intensity, axis=0)
incident_sum = float(np.sum(incident_column_power))
sim_transmission = []
for offset_um in transmission_offsets_um:
    exit_column_power = incident_column_power * np.exp(
        -absorption_coefficient_nm * analytic_wire_chord_nm(float(offset_um))
    )
    sim_transmission.append(float(np.sum(exit_column_power)) / incident_sum)
sim_transmission = np.asarray(sim_transmission)
sim_transmission /= sim_transmission.max()
np.savetxt(
    EXPERIMENTAL_TABLE_DIR / "analytic_exit_transmission.dat",
    np.column_stack((transmission_offsets_um, sim_transmission)),
    header="offset_um normalized_analytic_chord_target_exit_transmission",
)

del comparison_field_at_target, incident_intensity
gc.collect()
print("Experimental-comparison data complete")


Experimental comparison data 1/43
Experimental comparison data 2/43
Experimental comparison data 3/43
Experimental comparison data 4/43
Experimental comparison data 5/43
Experimental comparison data 6/43
Experimental comparison data 7/43
Experimental comparison data 8/43
Experimental comparison data 9/43
Experimental comparison data 10/43
Experimental comparison data 11/43
Experimental comparison data 12/43
Experimental comparison data 13/43
Experimental comparison data 14/43
Experimental comparison data 15/43
Experimental comparison data 16/43
Experimental comparison data 17/43
Experimental comparison data 18/43
Experimental comparison data 19/43
Experimental comparison data 20/43
Experimental comparison data 21/43
Experimental comparison data 22/43
Experimental comparison data 23/43
Experimental comparison data 24/43
Experimental comparison data 25/43
Experimental comparison data 26/43
Experimental comparison data 27/43
Experimental comparison data 28/43
Experimental comparison data 

## Completion summary


In [15]:
required_outputs = {
    "clean_saxs_tables": [
        *[case["table_path"] for case in CLEAN_SAXS_CASES.values()],
        PLOT_DATA_DIR / "clean_saxs_profiles.npz",
    ],
    "methods_tables": [
        methods_table_path, divergence_table_path,
        PLOT_DATA_DIR / "methods_pointing_profiles.npz",
        *methods_saxs_paths, *methods_pci_paths,
    ],
    "clean_pci": [
        *[case["fields_path"] for case in CLEAN_PCI_CASES.values()],
        *[case["detector_path"] for case in CLEAN_PCI_CASES.values()],
        *[case["table_path"] for case in CLEAN_PCI_CASES.values()],
        *[case["raw_table_path"] for case in CLEAN_PCI_CASES.values()],
        CLEAN_PCI_DIR / "mark1300_7p11um_shifted_q_mask_left_fraction.dat",
        CLEAN_PCI_DIR / "mark1300_6p0um_shifted_q_mask_left_fraction.dat",
    ],
    "realistic_profile": [
        PLOT_DATA_DIR / "realistic_incident_intensity.npy",
        realistic_pci_table_path, realistic_saxs_table_path, realistic_lineout_path,
    ],
    "experimental_comparison": [
        EXPERIMENTAL_TABLE_DIR / "analytic_exit_transmission.dat",
        *comparison_saxs_paths, *comparison_pci_paths,
    ],
}

summary = {
    "grid": N_GRID,
    "pixel_size_nm": float(PIXEL_SIZE / PCI.nm),
    "photon_energy_keV": float(PHOTON_ENERGY / PCI.keV),
    "sigma_tag": SIGMA_TAG,
    "wire_sigma_left_nm": WIRE_SIGMA_LEFT_NM,
    "wire_sigma_right_nm": WIRE_SIGMA_RIGHT_NM,
    "output_directory": str(OUTPUT_DIR.resolve()),
    "force_recalculate": FORCE_RECALCULATE,
    "sections": {name: [str(path.resolve()) for path in paths]
                 for name, paths in required_outputs.items()},
}
(PLOT_DATA_DIR / "calculation_manifest.json").write_text(
    json.dumps(summary, indent=2), encoding="utf-8"
)
print("Unified calculation pipeline complete")
print(f"Manifest: {PLOT_DATA_DIR / 'calculation_manifest.json'}")


Unified calculation pipeline complete
Manifest: c:\Users\feng48\Downloads\PCI_Codebase\PCI_Codebase\SAXS_asymmetry\outputs\sigmaL_4nm_sigmaR_4nm\combined_plot_data\calculation_manifest.json
